# BPL Autodiff and ML

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Slopes, walking downhill, and learning a line

Machine learning mostly means: pick a function with some adjustable numbers in it, measure how wrong it is, and nudge the numbers to make it less wrong. The nudging needs **slopes** (derivatives), and BPL works them out for you with one glyph, `∂`. This is called *automatic differentiation*, or *autodiff*.

This dialog starts with the slope of a simple function, then uses slopes to walk downhill to the bottom of a valley (*gradient descent*). It ends by teaching BPL to find the line through some points.

BPL docs: [Derivative `∂`](https://answerdotai.github.io/basedpl/glyphs/partial.html)

Glyphs and ideas this dialog teaches:

| Glyph | Plain English |
|---|---|
| `f∂x` | the slope (derivative) of `f` at `x` (typed **Alt-f**) |
| `f∂⍤0⁝v` | the slope at each item of `v` separately |
| gradient | with several inputs, `∂` gives one slope per input |
| gradient descent | take a small step downhill: `x - rate × slope` |
| learning rate | the size of that step, such as 0.1 |
| loss | one number saying how wrong a model is: smaller is better |
| `y⌹X` | (from linear algebra) the best-fit answer in one step, for checking |

Other glyphs used along the way:

| Glyph | Plain English |
|---|---|
| `←` | store a value or function in a name |
| `⍝` | comment: BPL ignores the rest of the line |
| `{ }`, `ω`, `α` | a dfn; its right and left arguments |
| `×⍨` | multiply a number by itself: square |
| `+/` `≢` | total, count |
| `[a b]←α` | store the two items of `α` in `a` and `b` |
| `◇` | separates statements on one line |
| `⍣` | repeat a function; `⍣(ιn)` keeps every step |
| `⁝` | keeps two things apart: the count and the start value |
| `¨` | each |
| `⍤0` | rank 0: one item at a time |
| `¿` `•prefs ["seed":3]` | random numbers, made repeatable |
| `⍉` `⊃` `ρ` | transpose, mix lists into a table, reshape |

### Slopes

#### 1. A slope, by hand

- `sq←×⍨` squares a number. At 3, it's 9.
- The *slope* at 3 says how fast `sq` is changing there. Measure it by hand: nudge 3 up by a tiny amount (0.001), and divide the change in the answer by the nudge.
- The answer is about 6.

Glyphs used here:

- `←` stores something in a name: `sq←×⍨` names the squaring function `sq`.
- `×` is multiply. `⍨` gives a function the same argument on both sides, so `×⍨ 3` means `3×3`.
- `⍝` starts a comment. BPL ignores the rest of the line.
- `-` and `÷` are subtract and divide.
- BPL works from right to left. In `((sq 3.001)-sq 3)÷0.001`, `sq 3` is worked out before the subtraction. The brackets around `sq 3.001` stop `sq` from squaring everything to its right.

In [4]:
%%bpl
sq←×⍨                       ⍝ square
sq 3                        ⍝ 9
sq 3.001
(sq 3.001)-sq 3
((sq 3.001)-sq 3)÷0.001     ⍝ slope by hand: about 6

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

9

9.006001

0.006001

6.001

#### 2. The exact slope: `∂`

- `sq∂3` gives the slope of `sq` at 3 directly: exactly 6. No nudging needed.
- `∂` works on any function you build from BPL's glyphs, dfns and trains included.
- To get the slope at several points, use `⍤0` so that `∂` looks at one point at a time. The slope of a square at `x` is always `2×x`.

In [5]:
%%bpl
sq∂3                 ⍝ exactly 6
sq∂⍤0⁝1 2 3          ⍝ slopes at 1, 2 and 3: 2 4 6

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

6

2 4 6

Glyph used here: `⍤` (rank).

- `f⍤0` applies `f` to each single number (rank-0 item) separately, then collects the results back into a list. So `sq∂⍤0⁝1 2 3` works out `sq∂1`, `sq∂2` and `sq∂3`.
- Without `⍤0`, `sq∂1 2 3` treats the whole list as *one* point with three inputs. It gives a 3×3 table of slopes (the *Jacobian*): how each output changes when each input is nudged. The answers `2 4 6` are on its diagonal, and the other entries are 0.

#### 3. One slope per input: the gradient

- `(+/sq) 1 2 3` squares each number and adds them up: `1+4+9` is 14. Three numbers go in, and one number comes out.
- Each input gets its own slope. To find the `3`'s slope, leave `1` and `2` alone and make only the `3` a tiny bit bigger. Only the `3²` part changes, so the slope is the slope of `sq` at 3, which is 6.
- Do the same for each input, and you get `2 4 6`. This list is called the *gradient*.
- Gradient descent (coming up) uses the gradient to decide which way to move each input.

In [6]:
%%bpl
(+/sq)∂1 2 3         ⍝ one slope per input: 2 4 6

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

2 4 6

### Walking downhill

#### 4. A valley

- `f←{sq ω-5}` is a valley with its lowest point at 5 (where `f` is 0).
- At 2, `f` is 9 and the slope is ¯6. A **negative** slope means the ground goes **down** to the right, so we should move right.

In [7]:
%%bpl
f←{sq ω-5}           ⍝ lowest at 5
f 2                  ⍝ 9
f∂2                  ⍝ ¯6: downhill is to the right

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

9

¯6

#### 5. Step downhill, again and again

- One step: `ω-0.1×f∂ω`. Moving **against** the slope always goes downhill. 0.1 is the step size, called the *learning rate*.
- From 2, one step goes to 2.6. Repeat with `⍣`, and it settles at 5: the bottom of the valley.
- This is *gradient descent*, the core of training almost every machine-learning model.

In [8]:
%%bpl
step←{ω-0.1×f∂ω}     ⍝ one step downhill
step 2               ⍝ 2.6
step⍣10⁝2            ⍝ after 10 steps: closer to 5
step⍣50⁝2            ⍝ after 50 steps: about 5

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

2.6

4.677877453

4.999957183

#### 6. Steps that are too big

- With a learning rate of 1.1, each step **overshoots** the bottom and lands further up the other side.
- `⍣(ι6)` keeps the first 6 positions: they jump back and forth and get worse each time. Too big a learning rate is a classic training bug.

In [9]:
%%bpl
big←{ω-1.1×f∂ω}      ⍝ steps too big
big⍣(ι6)⁝2           ⍝ jumps further away each time

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

2 8.6 0.68 10.184 ¯1.2208 12.46496

### Learning a line

#### 7. Some points, and a line

- Five points: `x` is 1 to 5, and `y` happens to be `1+2×x`. We'll pretend we don't know that, and let BPL find the 1 and the 2.
- A line has two numbers, a starting value `a` and a slope `b`: `a+b×x`. `line` takes them on the left, as a pair.
- `1 2 line x` gives exactly `y`.

In [10]:
%%bpl
x←1 2 3 4 5
y←3 5 7 9 11
line←{[a b]←α ◇ a+b×ω}     ⍝ a + b×x
1 2 line x                 ⍝ the right line gives y

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

3 5 7 9 11

#### 8. How wrong is a line? The loss

- The *loss* is one number saying how wrong a pair `a b` is. Here it's the mean of the squared errors.
- Each part goes in a name: `pred` (the line's guesses), `err` (how far off they are), then the mean of `sq err`. Separate names also keep BPL's spacing rules from splitting the line in the wrong places.
- The line `0 0` is very wrong (57), and `1 2` is perfect (0).

In [11]:
%%bpl
loss←{pred←ω line x ◇ err←y-pred ◇ (+/sq err)÷≢x}
loss 0 0             ⍝ 57: very wrong
loss 1 2             ⍝ 0: perfect

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

57

0

**Walkthrough: `loss←{pred←ω line x ◇ err←y-pred ◇ (+/sq err)÷≢x}`**

- `loss←{ … }` makes a function called `loss`. Inside it, `ω` is whatever you give it: here, a pair `a b`.
- `◇` separates three small steps. They run left to right, one after another. The value of the last step is the answer.

Step by step, using `loss 0 0`:

1. `pred←ω line x`: the line's guesses. `0 0 line x` means `0+0×x` at each `x`, so `pred` is `0 0 0 0 0`.
2. `err←y-pred`: how far off each guess is. `3 5 7 9 11` minus `0 0 0 0 0` gives `3 5 7 9 11`.
3. `(+/sq err)÷≢x`, read from the inside out:
   - `sq err` squares each error: `9 25 49 81 121`. Squaring turns a miss *below* the line into a positive number too, so misses can't cancel each other out. It also makes big misses count for much more than small ones.
   - `+/` adds them up: `285`.
   - `≢x` counts the points: `5`.
   - `÷` divides: `285÷5` is `57`. This is the average squared error.
   - The brackets matter. Without them, BPL would read right to left and work out `err÷≢x` first.

With `loss 1 2`, `pred` is `3 5 7 9 11`, which is exactly `y`. Every error is `0`, so the loss is `0`.

Smaller loss means a better line. Gradient descent will now walk `a b` downhill on this loss.

#### 9. Which way to nudge? The gradient

- `loss∂0 0` gives **two** slopes, one for `a` and one for `b`. Both are negative, so the loss goes down if we increase both.

In [12]:
%%bpl
loss∂0 0             ⍝ one slope for a, one for b

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

¯14 ¯50

#### 10. Train

- A training step is the same as step 5: move against the gradient, with a learning rate of 0.05.
- Starting from `0 0`, 500 steps find `a` and `b` very close to 1 and 2, and the loss is almost 0.

In [13]:
%%bpl
train←{ω-0.05×loss∂ω}      ⍝ one training step
p←train⍣500⁝0 0            ⍝ 500 steps from 0 0
p                          ⍝ about 1 2
loss p                     ⍝ almost 0

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.9999177665 2.000022777

1.230865657ₑ¯9

#### 11. Watch the loss fall, or blow up

- Keep the first 6 steps with `⍣(ι6)` and work out the loss of each with `loss¨`. It falls quickly.
- With a learning rate of 0.1, the loss **grows** at every step. If you see that, make the learning rate smaller.

In [14]:
%%bpl
loss¨train⍣(ι6)⁝0 0        ⍝ falls: 57 1.94 0.09 …
blow←{ω-0.1×loss∂ω}        ⍝ learning rate too big
loss¨blow⍣(ι6)⁝0 0         ⍝ grows: 57 106 198 …

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

57 1.94 0.0932 0.030344 0.02732432 0.0263411744

57 106.36 198.4912 370.453696 691.4192819 1290.495469

#### 12. Noisy data, checked another way

- Real data is never exactly on a line. `¿5ρ0` gives 5 random numbers between 0 and 1, so `yn` is `y` with a little noise added (the seed makes it repeatable).
- Gradient descent finds the best line for the noisy points.
- For a straight line, linear algebra can also find the best fit in **one** step: `yn⌹X`, where `X` has a column of 1s (for `a`) and a column of `x` (for `b`). The two answers match.
- Gradient descent is slower here, but it also works for models that have no one-step formula, like neural networks.

In [15]:
%%bpl
•prefs ["seed":3]
yn←y+¯0.5+¿5ρ0                      ⍝ y plus a little noise
yn
lossn←{pred←ω line x ◇ err←yn-pred ◇ (+/sq err)÷≢x}
trainn←{ω-0.05×lossn∂ω}
trainn⍣2000⁝0 0                     ⍝ gradient descent
ones←5ρ1
X←⍉⊃[ones x]                        ⍝ a column of 1s and a column of x
yn⌹X                                ⍝ one-step best fit: the same

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

3.122967464 4.581301697 6.822805663 8.583637762 11.09599073

0.8568258824 1.994838261

0.8568258824 1.994838261

### Your turn

1. Find the slope of `{ω×ω×ω}` (cube) at 2. (It should be 12.)
2. The valley `{sq ω+3}` has its lowest point at ¯3. Find it with gradient descent, starting at 0.
   *Hint:* copy step 5.
3. Train the line with a learning rate of 0.01 for 500 steps. How close does it get, compared with 0.05?
4. Change `y` to `2 4 6 8 10` and train again. What line do you find?
   *Hint:* run step 8's `loss` cell again after changing `y`.
5. How many steps does `train` take to get the loss below 0.001?
   *Hint:* `¯1+≢train⍣[{0.001>loss α}]⁝0 0` keeps every step until the test is true, then counts them.